# One-dimensional nearest-neighbor tight binding

**Prerequisite:** [One-dimensional nearest-neighbor tight binding](../../../docs/lecturenotes/solidstate/tight-binding/one-dimensional-nearest-neighbor/index.md)

This maintained laboratory constructs a scalar tight-binding Hamiltonian on a finite periodic index domain. It compares the matrix eigenvalues with the analytically sampled Bloch dispersion for periodic and twisted boundary conditions.

The parameters below are an **illustrative example**. They are not fitted material parameters or calculated first-principles results.

## Learning objectives

1. Define an ordered scalar hopping inventory with explicit units and energy reference.
2. Construct finite periodic operators in centered uniform-link gauge.
3. Compare represented matrix eigenvalues with the sampled Bloch dispersion.
4. Interpret a boundary twist as a shift of the sampled wave numbers.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.periodic.lattice.boundary_phases import (
    BoundaryTwistLift,
)
from projectkoios.physkit.periodic.lattice.finite_domain import (
    FinitePeriodicDomain,
    LatticeDimension,
    LatticeDisplacement,
)
from projectkoios.physkit.periodic.lattice.hopping import (
    ScalarHoppingModel,
    ScalarHoppingTerm,
)
from projectkoios.physkit.periodic.lattice.operator_construction import (
    TwistedSupercellOperatorConstructor,
)
from projectkoios.physkit.units import PhysicalUnit

In [ ]:
dimension = LatticeDimension.ONE
onsite_energy_ev = -2.0
hopping_magnitude_ev = 0.6
site_count = 16

model = ScalarHoppingModel(
    identifier="illustrative_nearest_neighbor_chain",
    dimension=dimension,
    terms=(
        ScalarHoppingTerm(
            LatticeDisplacement(dimension, (-1,)),
            -hopping_magnitude_ev,
            0.0,
        ),
        ScalarHoppingTerm(
            LatticeDisplacement(dimension, (0,)),
            onsite_energy_ev,
            0.0,
        ),
        ScalarHoppingTerm(
            LatticeDisplacement(dimension, (1,)),
            -hopping_magnitude_ev,
            0.0,
        ),
    ),
    energy_unit=PhysicalUnit("electron_volt"),
    energy_reference="illustrative_zero",
    basis_identifier="one_scalar_orbital_per_site",
)
domain = FinitePeriodicDomain(dimension, (site_count,))
constructor = TwistedSupercellOperatorConstructor()

In [ ]:
periodic_twist_turns = 0.0
shifted_twist_turns = 0.25

periodic_operator = constructor.execute(
    "periodic_ring",
    model,
    domain,
    BoundaryTwistLift(dimension, (periodic_twist_turns,)),
)
shifted_operator = constructor.execute(
    "quarter_twist_ring",
    model,
    domain,
    BoundaryTwistLift(dimension, (shifted_twist_turns,)),
)

periodic_matrix = periodic_operator.matrix.to_csr().toarray()
shifted_matrix = shifted_operator.matrix.to_csr().toarray()
periodic_eigenvalues = np.linalg.eigvalsh(periodic_matrix)
shifted_eigenvalues = np.linalg.eigvalsh(shifted_matrix)

In [ ]:
mode_indices = np.arange(site_count, dtype=np.float64)
periodic_wave_numbers = 2.0 * np.pi * (mode_indices + periodic_twist_turns) / site_count
shifted_wave_numbers = 2.0 * np.pi * (mode_indices + shifted_twist_turns) / site_count

periodic_expected = np.sort(
    onsite_energy_ev - 2.0 * hopping_magnitude_ev * np.cos(periodic_wave_numbers)
)
shifted_expected = np.sort(
    onsite_energy_ev - 2.0 * hopping_magnitude_ev * np.cos(shifted_wave_numbers)
)

assert np.allclose(periodic_matrix, periodic_matrix.conj().T)
assert np.allclose(shifted_matrix, shifted_matrix.conj().T)
assert np.allclose(
    periodic_eigenvalues,
    periodic_expected,
    rtol=0.0,
    atol=2e-15,
)
assert np.allclose(
    shifted_eigenvalues,
    shifted_expected,
    rtol=0.0,
    atol=2e-15,
)
assert np.isclose(
    np.trace(periodic_matrix).real,
    site_count * onsite_energy_ev,
)

maximum_periodic_spectral_error = float(
    np.max(np.abs(periodic_eigenvalues - periodic_expected))
)
maximum_shifted_spectral_error = float(
    np.max(np.abs(shifted_eigenvalues - shifted_expected))
)
maximum_periodic_spectral_error, maximum_shifted_spectral_error

In [ ]:
continuous_wave_numbers = np.linspace(-np.pi, np.pi, 600)
continuous_dispersion = onsite_energy_ev - 2.0 * hopping_magnitude_ev * np.cos(
    continuous_wave_numbers
)
wrapped_periodic_wave_numbers = (periodic_wave_numbers + np.pi) % (2.0 * np.pi) - np.pi
wrapped_shifted_wave_numbers = (shifted_wave_numbers + np.pi) % (2.0 * np.pi) - np.pi

figure, axis = plt.subplots(figsize=(9, 5))
axis.plot(
    continuous_wave_numbers / np.pi,
    continuous_dispersion,
    label="continuous Bloch dispersion",
)
axis.scatter(
    wrapped_periodic_wave_numbers / np.pi,
    onsite_energy_ev - 2.0 * hopping_magnitude_ev * np.cos(periodic_wave_numbers),
    marker="o",
    label="periodic finite-domain samples",
)
axis.scatter(
    wrapped_shifted_wave_numbers / np.pi,
    onsite_energy_ev - 2.0 * hopping_magnitude_ev * np.cos(shifted_wave_numbers),
    marker="x",
    label="quarter-twist samples",
)
axis.set(
    xlabel=r"dimensionless wave number $k$ divided by $\pi$",
    ylabel="energy (eV)",
    title="Nearest-neighbor tight-binding dispersion",
)
axis.grid(True)
axis.legend()
figure.tight_layout()
plt.show()

## Interpretation

The matrix eigensolver agrees with the analytically sampled dispersion to binary64 rounding. A finite periodic domain selects discrete wave numbers from the continuous band. The quarter-turn boundary twist shifts those sampling points without changing the underlying hopping model.

The finite-domain object represents integer site indices and periodic identification. It does not supply a physical lattice constant or atomic geometry. The horizontal axis therefore uses the dimensionless phase accumulated per lattice step.

## Exercises

1. Increase the site count and observe how the discrete samples fill the same band.
2. Change the hopping magnitude and verify that the bandwidth is $4|t|$.
3. Change the onsite energy and verify that the entire band shifts by the same amount.
4. Sweep the twist from zero to one turn and track one continuously labeled eigenvalue.
